In [3]:
import pandas as pd
import numpy as np
import time
import os
import warnings

from sklearn.model_selection import train_test_split
from ctgan import CTGAN

warnings.filterwarnings("ignore")

# Konfigurasi Path dan Variable
DATA_PATH = 'java-ash-hysplit.csv'
TARGET_PER_VOLCANO = 2000
RANDOM_STATE = 42
VOLCANO_COL = 'volcano_filter' # Ubah sesuai format nama kolom di dataset Anda
TARGET_COLS = ['jarak_km', 'luas_km2', 'sudut_deg', 'radius_km']

print("Dependencies berhasil dimuat.")

Dependencies berhasil dimuat.


In [4]:
start_prep = time.perf_counter()

# Membaca data
df = pd.read_csv(DATA_PATH)

# Memisahkan Hold-out Test set (20%)
train_val, test_df = train_test_split(
    df, 
    test_size=0.2, 
    random_state=RANDOM_STATE, 
    stratify=df[VOLCANO_COL]
)

# Memisahkan Train (60%) dan Val (20%) dari sisa train_val
# Untuk mendapatkan 20% dari total 100%, digunakan porsi 0.25 (karena 0.25 * 0.80 = 0.20)
train_df, val_df = train_test_split(
    train_val, 
    test_size=0.25, 
    random_state=RANDOM_STATE, 
    stratify=train_val[VOLCANO_COL]
)

end_prep = time.perf_counter()

print("Bentuk Data Setelah Split:")
print(f"Train set: {train_df.shape[0]} baris")
print(f"Val set  : {val_df.shape[0]} baris")
print(f"Test set : {test_df.shape[0]} baris")
print(f"Waktu persiapan: {end_prep - start_prep:.4f} detik")

Bentuk Data Setelah Split:
Train set: 1023 baris
Val set  : 341 baris
Test set : 342 baris
Waktu persiapan: 0.0176 detik


In [5]:
def apply_physical_constraints(df_synthetic):
    """
    Menyesuaikan kembali nilai sintetik yang keluar jalur dari batasan fisika logis.
    """
    df_clipped = df_synthetic.copy()
    
    # 1. Tidak boleh bernilai negatif untuk ukuran dimensi/jarak
    for col in ['jarak_km', 'luas_km2', 'radius_km']:
        if col in df_clipped.columns:
            df_clipped[col] = df_clipped[col].clip(lower=0.0)
            
    # 2. Sudut harus terkunci pada derajat 0-360
    if 'sudut_deg' in df_clipped.columns:
        df_clipped['sudut_deg'] = df_clipped['sudut_deg'] % 360.0
        
    return df_clipped

def get_discrete_columns(df):
    """
    Mengambil kolom yang direpresentasikan sebagai objek/kategorikal
    untuk dimasukkan sebagai identifikasi discrete_columns ke CTGAN.
    """
    discrete_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    return discrete_cols

In [6]:
start_total_gan = time.perf_counter()

discrete_columns = get_discrete_columns(train_df)
augmented_train_dfs = []
volcanoes = train_df[VOLCANO_COL].unique()

print("Memulai proses augmentasi CTGAN...")
print("-" * 50)

for v in volcanoes:
    start_v = time.perf_counter()
    
    # Filter dataset spesifik gunung
    df_v = train_df[train_df[VOLCANO_COL] == v]
    current_count = len(df_v)
    needed_count = TARGET_PER_VOLCANO - current_count
    
    print(f"Gunung: {v}")
    print(f"Status Data -> Asli: {current_count} | Kurang (Sintetik yang dibutuhkan): {max(0, needed_count)}")
    
    if needed_count <= 0:
        print("Telah mencapai/melebihi target, lewati augmentasi.")
        augmented_train_dfs.append(df_v)
        continue
        
    print(f"Tahap 1: Training Model CTGAN untuk {v}")
    # Epochs dapat ditingkatkan (e.g., 300) untuk model yang lebih presisi pada mesin dengan GPU
    ctgan_model = CTGAN(epochs=150, batch_size=500, verbose=False) 
    ctgan_model.fit(df_v, discrete_columns)
    
    print(f"Tahap 2: Generasi {needed_count} data sintetik")
    synthetic_data = ctgan_model.sample(needed_count)
    
    # Menjalankan constrain logika empiris abu vulkanik
    synthetic_data = apply_physical_constraints(synthetic_data)
    
    # Menggabungkan data train asli spesifik-gunung dengan data sintetik yang barusan digenerate
    combined_v = pd.concat([df_v, synthetic_data], ignore_index=True)
    augmented_train_dfs.append(combined_v)
    
    end_v = time.perf_counter()
    print(f">> Augmentasi {v} Selesai dalam waktu {end_v - start_v:.2f} detik")
    print("-" * 50)

# Merangkai kembali menjadi satu train dataset teraugmentasi utuh
df_train_augmented = pd.concat(augmented_train_dfs, ignore_index=True)

# Melakukan pengacakan baris sebelum disimpan (Shuffling)
df_train_augmented = df_train_augmented.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

end_total_gan = time.perf_counter()
print(f"TOTAL WAKTU RANGKAIAN AUGMENTASI: {end_total_gan - start_total_gan:.2f} detik")


Memulai proses augmentasi CTGAN...
--------------------------------------------------
Gunung: Semeru
Status Data -> Asli: 941 | Kurang (Sintetik yang dibutuhkan): 1059
Tahap 1: Training Model CTGAN untuk Semeru
Tahap 2: Generasi 1059 data sintetik
>> Augmentasi Semeru Selesai dalam waktu 28.42 detik
--------------------------------------------------
Gunung: Raung
Status Data -> Asli: 40 | Kurang (Sintetik yang dibutuhkan): 1960
Tahap 1: Training Model CTGAN untuk Raung
Tahap 2: Generasi 1960 data sintetik
>> Augmentasi Raung Selesai dalam waktu 8.77 detik
--------------------------------------------------
Gunung: Bromo
Status Data -> Asli: 16 | Kurang (Sintetik yang dibutuhkan): 1984
Tahap 1: Training Model CTGAN untuk Bromo
Tahap 2: Generasi 1984 data sintetik
>> Augmentasi Bromo Selesai dalam waktu 8.73 detik
--------------------------------------------------
Gunung: Merapi
Status Data -> Asli: 26 | Kurang (Sintetik yang dibutuhkan): 1974
Tahap 1: Training Model CTGAN untuk Merapi
Ta

In [7]:
print("Distribusi Kelas per Gunung di Data Train:")
print(df_train_augmented[VOLCANO_COL].value_counts())

start_save = time.perf_counter()

# Simpan ke storage untuk digunakan pipeline modelling (`modeling\`)
df_train_augmented.to_csv('java_ash_train_augmented.csv', index=False)
val_df.to_csv('java_ash_val.csv', index=False)
test_df.to_csv('java_ash_test.csv', index=False)

end_save = time.perf_counter()
print(f"Menyimpan CSV selesai dalam {end_save - start_save:.2f} detik.")

Distribusi Kelas per Gunung di Data Train:
volcano_filter
Raung     2000
Semeru    2000
Bromo     2000
Merapi    2000
Name: count, dtype: int64
Menyimpan CSV selesai dalam 0.15 detik.
